# TealKit MCP Agent — Git MCP — Qwen2.5-1.5B Training Pipeline

Fine-tunes **Qwen2.5-1.5B-Instruct** (`unsloth/Qwen2.5-1.5B-Instruct`) for Git/Search MCP tool-calling via Unsloth LoRA on Colab.

**Tool Set:** `index_websites`, `reindex_websites`, `purge_stale_index`, `list_indexed_pages`, `search_indexed_websites`, `get_indexed_page`

**Key Features:**
1. ChatML format with Qwen2.5-1.5B.
2. Configurable context window (8K/16K/32K/64K).
3. Loss-masking for assistant-only training.
4. Robust GGUF export with fallback.

## Cell 1 — Install Dependencies

In [ ]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" trl peft accelerate bitsandbytes datasets huggingface_hub

import shutil
shutil.rmtree('/root/.unsloth', ignore_errors=True)

print('Install done. Restarting runtime...')
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## Cell 2 — Config
> Set model, paths, and context window. Adjust `MAX_SEQ_LENGTH` based on your GPU memory.

In [ ]:
MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct'

# ── Context Window ──────────────────────────────────────────────────────────────
# Supported values for Qwen2.5-1.5B-Instruct:
#   8192   (8K)    — fits any GPU (T4/L4/H100)
#   16384  (16K)   — fits L4/H100, default
#   32768  (32K)   — native max, needs H100 or reduced batch size
#   65536  (64K)   — requires RoPE extension (YaRN), H100 recommended
MAX_SEQ_LENGTH = 16384

DATA_DIR = '/content/drive/MyDrive/Tealkit/training/git/mcp_out'
OUTPUT_DIR = '/content/drive/MyDrive/Tealkit/training/git/mcp_adapters_qwen25_1p5b'
GGUF_DIR = '/content/drive/MyDrive/Tealkit/training/git/mcp_fused_model_qwen25_1p5b'
MERGE_DIR = '/content/drive/MyDrive/Tealkit/training/git/mcp_merged_model_qwen25_1p5b'
SYSTEM_PROMPT_FILE = '/content/drive/MyDrive/Tealkit/training/git/git_system_prompt.md'
PREFER_EMBEDDED_UPDATED_PROMPT = True

# Hugging Face repo for upload (Cell 10)
HF_REPO = 'lschaffer/qwen25-1p5b-git'  # <-- Change to your HF username/repo

TRAIN_FILE = f'{DATA_DIR}/train_split.jsonl'
VALID_FILE = f'{DATA_DIR}/valid_split.jsonl'

print('Model                 :', MODEL_NAME)
print('MAX_SEQ_LENGTH        :', MAX_SEQ_LENGTH)
print('Train file            :', TRAIN_FILE)
print('Valid file            :', VALID_FILE)
print('Adapters out          :', OUTPUT_DIR)
print('GGUF out              :', GGUF_DIR)
print('HF repo               :', HF_REPO)


## Cell 3 — Mount Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)

for _path, _label in [(TRAIN_FILE, 'train_split.jsonl'), (VALID_FILE, 'valid_split.jsonl'), (SYSTEM_PROMPT_FILE, 'git system prompt')]:
    if not os.path.isfile(_path):
        print(f'MISSING {_label}: {_path} -> Upload to Drive first!')
    else:
        print(f'OK  {_label} found at {_path}')

## Cell 4 — Load model with Native bfloat16

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   
    dtype=torch.bfloat16, 
)
print('Model loaded in native bfloat16 precision.')

## Cell 5 — Apply LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
model.print_trainable_parameters()

## Cell 6 — Load dataset + System Prompt

In [ ]:
from datasets import load_dataset
import json
import os
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template='chatml',
)

UPDATED_SYSTEM_PROMPT = '''# ROLE
You are a Web Search Assistant with access to website indexing and semantic search tools.

# THE 3 GOLDEN RULES
1. TOOL USE ONLY: Use the provided tools for all indexing and search tasks.
2. SILENT TOOL CALLS: Output only the canonical tool call when a tool is needed.
3. FINAL ANSWERS ONLY AFTER DATA: Answer directly from tool results.

# CANONICAL TOOL CALL FORMAT
- Use exactly: `tool_call: {"name":"<tool_name>","arguments":{...}}`
- Do not add explanation before or after the tool call.

# CURRENT SCHEMA TOOLS
- `index_websites`: Crawl and index configured websites. No parameters.
- `reindex_websites`: Rebuild the website index. No parameters.
- `purge_stale_index`: Delete stale indexed rows. No parameters.
- `list_indexed_pages`: List indexed pages. Params: domain (opt), limit (default 50).
- `search_indexed_websites`: Search indexed content. Required: query. Optional: domain, limit, searchMode.
- `get_indexed_page`: Get full content for a URL. Required: url.

# ORDER OF OPERATIONS
1. Search first for info retrieval.
2. Use domain filter if user specifies a website.
3. Use index_websites for crawling, reindex_websites for rebuilding.

# FINAL ANSWER STYLE
- Keep answers short, direct, factual.
- Summarize search results with page titles/URLs.
- Do not suggest extra work unless explicitly asked.'''

def is_modern_system_prompt(text):
    required_markers = [
        '# FINAL ANSWER STYLE',
        'search_indexed_websites',
        'list_indexed_pages',
    ]
    return all(marker in text for marker in required_markers)

if os.path.isfile(SYSTEM_PROMPT_FILE):
    with open(SYSTEM_PROMPT_FILE, 'r', encoding='utf-8') as f:
        DRIVE_SYSTEM_PROMPT = f.read().strip()
    if PREFER_EMBEDDED_UPDATED_PROMPT and not is_modern_system_prompt(DRIVE_SYSTEM_PROMPT):
        SYSTEM_PROMPT = UPDATED_SYSTEM_PROMPT
        print('Drive system prompt is stale. Using embedded updated prompt instead.')
    else:
        SYSTEM_PROMPT = DRIVE_SYSTEM_PROMPT
        print('Loaded modern system prompt from Drive:', SYSTEM_PROMPT_FILE)
else:
    SYSTEM_PROMPT = UPDATED_SYSTEM_PROMPT
    print('Drive system prompt not found. Using embedded updated prompt.')

def format_example(examples):
    texts = []
    for msgs in examples['messages']:
        if not msgs or msgs[0].get('role') != 'system':
            msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}] + list(msgs)
        texts.append(
            tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False
            )
        )
    return {'text': texts}

dataset = load_dataset('json', data_files={'train': TRAIN_FILE, 'validation': VALID_FILE})
dataset = dataset.map(format_example, batched=True)

print('Train examples:', len(dataset['train']))
print('Valid examples:', len(dataset['validation']))

def detect_chat_markers(sample_text):
    instruction_candidates = [
        '<|im_start|>user\\n',
        '<|im_start|>user<|im_sep|>',
        '<|user|>',
    ]
    response_candidates = [
        '<|im_start|>assistant\\n',
        '<|im_start|>assistant<|im_sep|>',
        '<|assistant|>',
    ]
    instruction_part = next((part for part in instruction_candidates if part in sample_text), None)
    response_part = next((part for part in response_candidates if part in sample_text), None)
    return instruction_part, response_part

_sample_text = dataset['train'][0]['text'] if len(dataset['train']) else ''
INSTRUCTION_PART, RESPONSE_PART = detect_chat_markers(_sample_text)
print('Instruction marker:', repr(INSTRUCTION_PART))
print('Response marker   :', repr(RESPONSE_PART))


## Cell 7 — Training Loop

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only

_batch_size = 4 if MAX_SEQ_LENGTH <= 8192 else (2 if MAX_SEQ_LENGTH <= 16384 else 1)
_grad_accum = 2 if MAX_SEQ_LENGTH <= 8192 else (4 if MAX_SEQ_LENGTH <= 32768 else 8)

_args = SFTConfig(
    per_device_train_batch_size=_batch_size,
    gradient_accumulation_steps=_grad_accum,
    warmup_steps=5,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='no',
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=3407,
    output_dir='/content/outputs',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=_args,
)

if INSTRUCTION_PART and RESPONSE_PART:
    try:
        masked_trainer = train_on_responses_only(
            trainer,
            instruction_part=INSTRUCTION_PART,
            response_part=RESPONSE_PART,
        )
        if len(masked_trainer.train_dataset) == 0:
            print('WARNING: Response masking removed every train sample. Falling back to full-sequence training.')
        else:
            trainer = masked_trainer
            print(f'Response masking active: {len(trainer.train_dataset)} train samples.')
    except Exception as e:
        print("WARNING: Response masking failed, using full sequence. Error:", e)
else:
    print('WARNING: Could not detect chat markers in formatted text. Using full-sequence training.')

trainer.train()
print('Training complete.')

## Cell 8 — Save adapters + robust GGUF export

In [ ]:
import gc, glob, os, shutil, subprocess, sys, torch
if 'OUTPUT_DIR' not in dir(): OUTPUT_DIR = '/content/drive/MyDrive/Tealkit/training/git/mcp_adapters_qwen25_1p5b'
if 'GGUF_DIR' not in dir(): GGUF_DIR = '/content/drive/MyDrive/Tealkit/training/git/mcp_fused_model_qwen25_1p5b'
if 'MERGE_DIR' not in dir(): MERGE_DIR = '/content/drive/MyDrive/Tealkit/training/git/mcp_merged_model_qwen25_1p5b'
QUANT_METHOD = 'q4_k_m'
GGUF_BASENAME = f"{HF_REPO.split('/')[-1]}-unsloth"
GGUF_F16_PATH = os.path.join(GGUF_DIR, f'{GGUF_BASENAME}-F16.gguf')
GGUF_QUANT_PATH = os.path.join(GGUF_DIR, f'{GGUF_BASENAME}-{QUANT_METHOD.upper()}.gguf')
FINAL_GGUF_FILE = None
GGUF_FILENAME = None

os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Adapters saved to Drive:', OUTPUT_DIR)

def run_checked(command, cwd=None, extra_env=None):
    env = os.environ.copy()
    for key in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP', 'PYTHONUSERBASE'):
        env.pop(key, None)
    env['PYTHONNOUSERSITE'] = '1'
    if extra_env:
        for key, value in extra_env.items():
            if value is None:
                env.pop(key, None)
            else:
                env[key] = value
    print('>>', ' '.join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def clear_unsloth_llama_cpp_cache():
    d = '/root/.unsloth/llama.cpp'
    if os.path.isdir(d): shutil.rmtree(d, ignore_errors=True)

def refresh_tokenizer_files(merged_dir, base_model):
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        return
    for fn in ('tokenizer_config.json', 'tokenizer.json', 'special_tokens_map.json'):
        try:
            src = hf_hub_download(repo_id=base_model, filename=fn)
            shutil.copy2(src, os.path.join(merged_dir, fn))
        except Exception:
            pass

def find_quantize_binary(llama_cpp_dir):
    candidates = [
        os.path.join(llama_cpp_dir, 'build', 'bin', 'llama-quantize'),
        os.path.join(llama_cpp_dir, 'build', 'bin', 'quantize'),
        shutil.which('llama-quantize'), shutil.which('quantize'),
    ]
    for c in candidates:
        if c and os.path.isfile(c) and os.access(c, os.X_OK):
            return c
    return None

def ensure_quantize_binary(llama_cpp_dir):
    b = find_quantize_binary(llama_cpp_dir)
    if b: return b
    try:
        run_checked(['cmake', '-S', '.', '-B', 'build', '-DBUILD_SHARED_LIBS=OFF', '-DGGML_CUDA=OFF'], cwd=llama_cpp_dir)
        run_checked(['cmake', '--build', 'build', '--config', 'Release', '-j2'], cwd=llama_cpp_dir)
    except Exception:
        pass
    return find_quantize_binary(llama_cpp_dir)

def install_llama_cpp_requirements(llama_cpp_dir, pydeps_dir):
    req = os.path.join(llama_cpp_dir, 'requirements.txt')
    if os.path.isdir(pydeps_dir): shutil.rmtree(pydeps_dir)
    os.makedirs(pydeps_dir, exist_ok=True)
    if os.path.isfile(req):
        try:
            run_checked([sys.executable, '-m', 'pip', 'install', '--upgrade', '--target', pydeps_dir, '-r', req])
        except Exception:
            run_checked([sys.executable, '-m', 'pip', 'install', '--upgrade', '--target', pydeps_dir,
                'numpy', 'sentencepiece', 'protobuf', 'safetensors', 'transformers', 'huggingface_hub', 'tqdm'])

def ensure_llama_cpp_checkout(llama_cpp_dir):
    cs = os.path.join(llama_cpp_dir, 'convert_hf_to_gguf.py')
    if not os.path.isdir(os.path.join(llama_cpp_dir, '.git')):
        if os.path.exists(llama_cpp_dir): shutil.rmtree(llama_cpp_dir, ignore_errors=True)
        run_checked(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', llama_cpp_dir])
    return cs

def manual_llama_cpp_convert(merged_dir):
    clear_unsloth_llama_cpp_cache()
    refresh_tokenizer_files(merged_dir, MODEL_NAME)
    lcd = '/content/llama.cpp'; pd = '/content/llama_cpp_pydeps'
    cs = ensure_llama_cpp_checkout(lcd)
    gp = os.path.join(lcd, 'gguf-py')
    install_llama_cpp_requirements(lcd, pd)
    ipp = os.pathsep.join([p for p in [pd, gp, lcd] if p])
    qb = ensure_quantize_binary(lcd)
    try:
        run_checked([sys.executable, '-S', cs, merged_dir, '--outfile', GGUF_F16_PATH, '--outtype', 'f16'], cwd=lcd, extra_env={'PYTHONPATH': ipp})
    except subprocess.CalledProcessError:
        refresh_tokenizer_files(merged_dir, MODEL_NAME)
        cs = ensure_llama_cpp_checkout(lcd, force_fresh=True) if 'force_fresh' in dir() else ensure_llama_cpp_checkout(lcd)
        run_checked([sys.executable, '-S', cs, merged_dir, '--outfile', GGUF_F16_PATH, '--outtype', 'f16'], cwd=lcd, extra_env={'PYTHONPATH': ipp})
    if qb:
        run_checked([qb, GGUF_F16_PATH, GGUF_QUANT_PATH, QUANT_METHOD.upper()])
        return GGUF_QUANT_PATH
    return GGUF_F16_PATH

if os.path.exists(GGUF_DIR): shutil.rmtree(GGUF_DIR)
os.makedirs(GGUF_DIR, exist_ok=True)

try:
    clear_unsloth_llama_cpp_cache()
    trainer.model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method=QUANT_METHOD)
    _gf = sorted(glob.glob(f'{GGUF_DIR}/*.gguf'))
    if not _gf: raise RuntimeError(f'No GGUF in {GGUF_DIR}')
    FINAL_GGUF_FILE = _gf[0]; GGUF_FILENAME = os.path.basename(FINAL_GGUF_FILE)
    print('Native GGUF export complete:', FINAL_GGUF_FILE)
except Exception as e:
    print(f'Native GGUF failed: {e}. Falling back to manual conversion...')
    if os.path.exists(MERGE_DIR): shutil.rmtree(MERGE_DIR)
    trainer.model.save_pretrained_merged(MERGE_DIR, tokenizer, save_method='merged_16bit')
    FINAL_GGUF_FILE = manual_llama_cpp_convert(MERGE_DIR)
    GGUF_FILENAME = os.path.basename(FINAL_GGUF_FILE)

if not FINAL_GGUF_FILE or not os.path.isfile(FINAL_GGUF_FILE):
    raise RuntimeError('GGUF export failed.')

GGUF_FILENAME = os.path.basename(FINAL_GGUF_FILE)
print(f'Final GGUF: {FINAL_GGUF_FILE}')
torch.cuda.empty_cache(); gc.collect()

## Cell 9 — Generate Model Card

In [ ]:
import os

MODEL_CARD_PATH = f"{GGUF_DIR}/README.md"
GGUF_FILENAME = globals().get('GGUF_FILENAME') or f"{HF_REPO.split('/')[-1]}-unsloth-{QUANT_METHOD.upper()}.gguf"

model_card_content = f'''---
base_model: {MODEL_NAME}
library_name: unsloth
tags:
- mcp
- web-crawl
- search
- tool-calling
- gguf
---

# Git MCP Agent - {MODEL_NAME.split('/')[-1]}

Fine-tuned for Git/Search MCP tool set.

## Tools
- index_websites, reindex_websites, purge_stale_index
- list_indexed_pages, search_indexed_websites, get_indexed_page

## Files
- GGUF: {GGUF_FILENAME}
- Adapters: saved during notebook execution
'''

with open(MODEL_CARD_PATH, 'w', encoding='utf-8') as handle:
    handle.write(model_card_content)
print('Model card generated at:', MODEL_CARD_PATH)

UPLOAD_TO_HF = False
if UPLOAD_TO_HF:
    import getpass
    from huggingface_hub import HfApi, upload_folder
    try:
        hf_token = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
        api = HfApi(token=hf_token)
        api.create_repo(repo_id=HF_REPO, repo_type='model', exist_ok=True)
        upload_folder(repo_id=HF_REPO, folder_path=GGUF_DIR, repo_type='model', token=hf_token)
        print('Uploaded to HF:', HF_REPO)
    except Exception as exc:
        print('HF upload skipped:', exc)